In [1]:
import pandas as pd
import numpy as np
import joblib

from pathlib import Path

In [2]:
BASE_DIR = Path.cwd().parent

DATA_DIR = BASE_DIR / "data" / "processed"
FEATURE_DIR = DATA_DIR / "features"
MODEL_DIR = BASE_DIR / "models"

MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Model directory:", MODEL_DIR)

Model directory: /home/shahzad-mustafa/Desktop/MovieRecommendationSystem/models


In [4]:
weight_comparison = pd.read_csv(
    BASE_DIR / "reports" / "hybrid_weight_comparison.csv"
)

best_alpha = weight_comparison.loc[
    weight_comparison["Hybrid Precision@10"].idxmax(),
    "Alpha"
]

print("Selected hybrid alpha:", best_alpha)

Selected hybrid alpha: 0.7


In [5]:
movies = pd.read_csv(DATA_DIR / "movies_clean.csv")

ratings = pd.read_csv(DATA_DIR / "ratings_clean.csv")

genre_features = pd.read_csv(
    FEATURE_DIR / "movie_genre_features.csv",
    index_col="MovieID"
)

genre_features.index = genre_features.index.astype(int)

movies["MovieID"] = movies["MovieID"].astype(int)
ratings["MovieID"] = ratings["MovieID"].astype(int)
ratings["UserID"] = ratings["UserID"].astype(int)

print("Movies:", movies.shape)
print("Ratings:", ratings.shape)
print("Genre features:", genre_features.shape)

Movies: (3883, 3)
Ratings: (1000209, 5)
Genre features: (3883, 18)


In [6]:
user_movie_matrix = ratings.pivot(
    index="UserID",
    columns="MovieID",
    values="Rating"
)

user_means = user_movie_matrix.mean(axis=1)

print("User-movie matrix:", user_movie_matrix.shape)
print("User means:", user_means.shape)

User-movie matrix: (6040, 3706)
User means: (6040,)


In [7]:
from scipy.sparse import csr_matrix, lil_matrix

centered_matrix = (
    user_movie_matrix
    .sub(user_means, axis=0)
    .fillna(0)
)

centered_sparse = csr_matrix(
    centered_matrix.to_numpy(dtype=np.float32)
)

rating_presence = csr_matrix(
    user_movie_matrix.notna().to_numpy(dtype=np.float32)
)

print("Centered matrix:", centered_sparse.shape)
print("Rating presence:", rating_presence.shape)

Centered matrix: (6040, 3706)
Rating presence: (6040, 3706)


In [8]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(
    centered_sparse,
    dense_output=False
).tocsr()

print("Similarity matrix:", similarity_matrix.shape)

Similarity matrix: (6040, 6040)


In [9]:
top20_similarity = lil_matrix(
    similarity_matrix.shape,
    dtype=np.float32
)

for user_index in range(similarity_matrix.shape[0]):

    similarity_row = similarity_matrix.getrow(user_index)

    neighbor_indices = similarity_row.indices
    neighbor_scores = similarity_row.data

    valid_mask = (
        (neighbor_indices != user_index)
        & (neighbor_scores > 0)
    )

    neighbor_indices = neighbor_indices[valid_mask]
    neighbor_scores = neighbor_scores[valid_mask]

    if len(neighbor_scores) > 20:

        top_indices = np.argsort(neighbor_scores)[-20:]

        neighbor_indices = neighbor_indices[top_indices]
        neighbor_scores = neighbor_scores[top_indices]

    top20_similarity[user_index, neighbor_indices] = neighbor_scores

top20_similarity = top20_similarity.tocsr()

print("Top-20 similarity matrix:", top20_similarity.shape)

Top-20 similarity matrix: (6040, 6040)


In [10]:
weighted_deviations = (
    top20_similarity.dot(centered_sparse)
).tocsr()

similarity_weights = (
    top20_similarity.dot(rating_presence)
).tocsr()

print("Weighted deviations:", weighted_deviations.shape)
print("Similarity weights:", similarity_weights.shape)

Weighted deviations: (6040, 3706)
Similarity weights: (6040, 3706)


In [11]:
model_artifacts = {
    "movies": movies,
    "genre_features": genre_features,
    "user_movie_matrix": user_movie_matrix,
    "user_means": user_means,
    "top20_similarity": top20_similarity,
    "weighted_deviations": weighted_deviations,
    "similarity_weights": similarity_weights,
    "best_alpha": float(best_alpha)
}

print("Artifacts prepared:", list(model_artifacts.keys()))

Artifacts prepared: ['movies', 'genre_features', 'user_movie_matrix', 'user_means', 'top20_similarity', 'weighted_deviations', 'similarity_weights', 'best_alpha']


In [12]:
joblib.dump(
    model_artifacts,
    MODEL_DIR / "movie_recommender_artifacts.joblib",
    compress=3
)

print("Model artifacts saved successfully.")

Model artifacts saved successfully.


In [13]:
artifact_path = MODEL_DIR / "movie_recommender_artifacts.joblib"

print("File exists:", artifact_path.exists())
print("File size (MB):", round(artifact_path.stat().st_size / (1024 * 1024), 2))

File exists: True
File size (MB): 60.17


In [14]:
loaded_artifacts = joblib.load(artifact_path)

print("Loaded components:")

for key in loaded_artifacts:
    print("-", key)

Loaded components:
- movies
- genre_features
- user_movie_matrix
- user_means
- top20_similarity
- weighted_deviations
- similarity_weights
- best_alpha


In [15]:
artifacts = joblib.load(artifact_path)

movies = artifacts["movies"]
genre_features = artifacts["genre_features"]
user_movie_matrix = artifacts["user_movie_matrix"]
user_means = artifacts["user_means"]

top20_similarity = artifacts["top20_similarity"]
weighted_deviations = artifacts["weighted_deviations"]
similarity_weights = artifacts["similarity_weights"]

best_alpha = artifacts["best_alpha"]

In [16]:
def recommend_movies(user_id, top_n=10):

    if user_id not in user_movie_matrix.index:
        raise ValueError("User ID not found.")

    user_ratings = user_movie_matrix.loc[user_id]

    liked_movie_ids = user_ratings[
        user_ratings >= 4
    ].dropna().index

    if len(liked_movie_ids) == 0:
        return pd.DataFrame()

    # Build user genre profile
    available_movies = genre_features.index.intersection(
        liked_movie_ids
    )

    if len(available_movies) == 0:
        return pd.DataFrame()

    user_profile = genre_features.loc[
        available_movies
    ].mean(axis=0)

    # Content-based scores
    content_scores = cosine_similarity(
        user_profile.values.reshape(1, -1),
        genre_features
    )[0]

    content_scores = pd.Series(
        content_scores,
        index=genre_features.index
    )

    # Collaborative filtering scores
    user_index = user_movie_matrix.index.get_loc(user_id)

    numerator = weighted_deviations.getrow(
        user_index
    ).toarray().ravel()

    denominator = similarity_weights.getrow(
        user_index
    ).toarray().ravel()

    valid_mask = denominator > 0

    predicted_ratings = np.full(
        len(user_movie_matrix.columns),
        np.nan
    )

    predicted_ratings[valid_mask] = (
        user_means.loc[user_id]
        + numerator[valid_mask] / denominator[valid_mask]
    )

    predicted_ratings = np.clip(
        predicted_ratings,
        1,
        5
    )

    collaborative_scores = pd.Series(
        predicted_ratings,
        index=user_movie_matrix.columns
    )

    collaborative_scores = (
        (collaborative_scores - 1) / 4
    )

    # Combine both models
    common_ids = content_scores.index.intersection(
        collaborative_scores.dropna().index
    )

    results = pd.DataFrame({
        "MovieID": common_ids,
        "ContentScore": content_scores.loc[common_ids].values,
        "CollaborativeScore": collaborative_scores.loc[common_ids].values
    })

    results["HybridScore"] = (
        best_alpha * results["ContentScore"]
        + (1 - best_alpha) * results["CollaborativeScore"]
    )

    # Exclude watched movies
    watched_movie_ids = set(
        user_ratings.dropna().index
    )

    results = results[
        ~results["MovieID"].isin(watched_movie_ids)
    ]

    results = results.merge(
        movies[["MovieID", "Title", "Genres"]],
        on="MovieID",
        how="left"
    )

    return results.sort_values(
        "HybridScore",
        ascending=False
    ).head(top_n).reset_index(drop=True)

In [17]:
recommend_movies(user_id=1, top_n=10)

,MovieID,ContentScore,CollaborativeScore,HybridScore,Title,Genres
0,1012,0.765737,1.000000,0.836016,Old Yeller (1957),Children's|Drama
1,2102,0.763594,0.993978,0.832710,Steamboat Willie (1940),Animation|Children's|Comedy|Musical
2,262,0.765737,0.901336,0.806417,"Little Princess, A (1995)",Children's|Drama
3,364,0.705378,0.994618,0.792150,"Lion King, The (1994)",Animation|Children's|Musical
4,1032,0.705378,0.993978,0.791958,Alice in Wonderland (1951),Animation|Children's|Musical
5,34,0.801566,0.768025,0.791504,Babe (1995),Children's|Comedy|Drama
6,2059,0.765737,0.843324,0.789013,"Parent Trap, The (1998)",Children's|Drama
7,2080,0.720233,0.878045,0.767576,Lady and the Tramp (1955),Animation|Children's|Comedy|Musical|Romance
8,3159,0.705378,0.909845,0.766718,Fantasia 2000 (1999),Animation|Children's|Musical
9,2096,0.705378,0.903433,0.764795,Sleeping Beauty (1959),Animation|Children's|Musical


In [18]:
test_users = [1, 10, 100]

for user_id in test_users:

    print(f"\nRecommendations for User {user_id}")
    print("-" * 60)

    recommendations = recommend_movies(
        user_id=user_id,
        top_n=5
    )

    display(
        recommendations[
            ["Title", "ContentScore", "CollaborativeScore", "HybridScore"]
        ]
    )


Recommendations for User 1
------------------------------------------------------------


,Title,ContentScore,CollaborativeScore,HybridScore
0,Old Yeller (1957),0.765737,1.000000,0.836016
1,Steamboat Willie (1940),0.763594,0.993978,0.832710
2,"Little Princess, A (1995)",0.765737,0.901336,0.806417
3,"Lion King, The (1994)",0.705378,0.994618,0.792150
4,Alice in Wonderland (1951),0.705378,0.993978,0.791958



Recommendations for User 10
------------------------------------------------------------


,Title,ContentScore,CollaborativeScore,HybridScore
0,Cinema Paradiso (1988),0.754143,1.000000,0.827900
1,"Next Stop, Wonderland (1998)",0.754143,1.000000,0.827900
2,Twelfth Night (1996),0.754143,1.000000,0.827900
3,Brassed Off (1996),0.754143,0.989548,0.824764
4,"Other Sister, The (1999)",0.754143,0.981362,0.822309



Recommendations for User 100
------------------------------------------------------------


,Title,ContentScore,CollaborativeScore,HybridScore
0,Ben-Hur (1959),0.900997,0.826929,0.878777
1,"Emerald Forest, The (1985)",0.900997,0.659357,0.828505
2,Romper Stomper (1992),0.788208,0.782752,0.786572
3,Cyrano de Bergerac (1990),0.746541,0.754912,0.749052
4,Gladiator (2000),0.788208,0.639679,0.743649


In [19]:
try:
    recommend_movies(user_id=999999)

except ValueError as error:
    print("Validation successful:", error)

Validation successful: User ID not found.


In [20]:
movies[movies["Title"].str.contains("Titanic", case=False, na=False)]

,MovieID,Title,Genres
1672,1721,Titanic (1997),Drama|Romance
2088,2157,"Chambermaid on the Titanic, The (1998)",Romance
3334,3403,Raise the Titanic (1980),Drama|Thriller
3335,3404,Titanic (1953),Action|Drama


In [21]:
titanic_id = 1721

titanic_id in movies["MovieID"].values

True

In [22]:
movies[movies["Title"].str.contains("Avatar", case=False, na=False)]

,MovieID,Title,Genres
